In [ ]:
import os
import sys
import torch
import hydra
from omegaconf import OmegaConf
from datetime import datetime
from quantifydrivers.train_and_shap.config_schema import validate_schema

from pathlib import Path

# Detect if running in a Notebook or a standard script
try:
    # This works in scripts
    file_path = Path(__file__).resolve()
    script_dir = file_path.parent
except NameError:
    # This works in Notebooks
    script_dir = Path(os.getcwd()).resolve()

# Assuming the notebook/script is deep inside the project (e.g., quantifydrivers/train_and_shap/)
# Adjust .parent.parent as needed to reach the root containing 'quantifydrivers'
project_root = script_dir.parent.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"✅ Added to sys.path: {project_root}")
else:
    print(f"Path already exists: {project_root}")

from quantifydrivers.machine_learning.dataloading_script import build_datasets_and_loaders
from quantifydrivers.train_and_shap.ensemble_script import run_ensemble_analysis
# --- 1. PATH FIX FOR NOTEBOOKS ---
# Notebooks run in the current working directory.
# Assuming your notebook is in the same folder where the script would be (e.g., inside 'quantifydrivers/train_and_shap/')
cwd = os.getcwd()

# If your notebook is in the root, adjust this.
# This logic tries to find the project root relative to where the notebook is running.
project_src_dir = os.path.abspath(os.path.join(cwd, '..', '..'))

if project_src_dir not in sys.path:
    sys.path.append(project_src_dir)
    print(f"Added to sys.path: {project_src_dir}")

# --- 2. LOAD CONFIG (Hydra Compose API) ---
# This replaces @hydra.main
try:
    hydra.initialize(version_base=None, config_path="conf")
except ValueError:
    # This prevents the error if you re-run the cell (Hydra can only be initialized once)
    pass

# Load the config and override values if needed
cfg = hydra.compose(config_name="config", overrides=["site=cordoba", "percentile=90p"])

print("*** Loaded Config ***")
print(OmegaConf.to_yaml(cfg))

# --- 3. RUN PIPELINE ---
# Global Settings
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
timestamp = datetime.now().strftime('%m-%d-%Y--%H-%M')

print(f"*** Starting Ensemble Pipeline at {timestamp} ***")
print(f"Site: {cfg.site} | Percentile: {cfg.percentile}")

# Validate Config
try:
    validated_cfg = validate_schema(cfg)
    print("✅ Config validation passed!")
except Exception as e:
    print(f"❌ Config Validation Error: {e}")
    raise

# Create generator
g = torch.Generator()
g.manual_seed(validated_cfg.SEED)

# Build Datasets
datasets = build_datasets_and_loaders(configuration=validated_cfg, generator=g)

# Run Analysis
from ensemble_script import run_ensemble_analysis
run_ensemble_analysis(
    configuration=validated_cfg,
    datasets=datasets,
    device=device,
    timestamp=timestamp
)